# 😊 Yelp Review Sentiment — Interactive Colab


In [ ]:
#@title 🚀 Setup (run once)
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import output
    output.enable_custom_widget_manager()
try:
    import ipywidgets as widgets
except Exception:
    !pip -q install ipywidgets
    import ipywidgets as widgets
try:
    import sklearn, pandas, numpy, matplotlib, joblib, seaborn
except Exception:
    !pip -q install scikit-learn pandas numpy matplotlib joblib seaborn
    import sklearn, pandas, numpy, matplotlib, joblib, seaborn
from IPython.display import display, Markdown
display(Markdown('**✅ Setup complete.**'))

In [2]:
#@title 📝 Reflection helpers (run once)
import time, csv
import ipywidgets as widgets
from IPython.display import display, Markdown
try:
    _REFLECTIONS
except NameError:
    _REFLECTIONS = {}
def reflection_box(prompt: str, placeholder: str = 'Type your thoughts here...'):
    title = widgets.HTML(value=f"<b>Reflection:</b> {prompt}")
    ta = widgets.Textarea(value=_REFLECTIONS.get(prompt, ''), placeholder=placeholder,
                          layout={'width':'100%','height':'100px'})
    btn = widgets.Button(description='Save note')
    out = widgets.Output()
    def save(_):
        _REFLECTIONS[prompt] = ta.value
        with out:
            out.clear_output(); display(Markdown('**Saved.** Use export_reflections() to download.'))
    btn.on_click(save)
    display(widgets.VBox([title, ta, btn, out]))
def export_reflections(path='yelp_reflections.csv'):
    with open(path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f); writer.writerow(['question','response'])
        for q, a in _REFLECTIONS.items(): writer.writerow([q, a])
    return path
display(Markdown('**Reflection helpers ready.**'))

**Reflection helpers ready.**

In [3]:
#@title Reflection: What do you think this notebook will help you learn?
reflection_box("In one or two sentences, what do you think this notebook will help you learn?")

## 1) Load Yelp data

In [10]:
#@title Reflection: Understanding the Dataset
reflection_box("What do you notice about this dataset? What do you think the 'text' and 'sentiment' columns represent?")

In [5]:
#@title Reflection: Features in the Dataset
reflection_box("How many features (columns) are there in this dataset? What do you think 'features' means in machine learning?")

In [6]:
#@title Reflection: Classifying Reviews Yourself
reflection_box("If you had to classify a review as positive or negative yourself, what words or phrases would you look for?")

In [7]:
#@title Reflection: Data Cleaning
reflection_box("What do you think this cleaning code is doing? Why do you think we need to normalize labels and remove duplicates?")

In [8]:
#@title Reflection: Data Cleaning Process
reflection_box("How is this data cleaning similar to or different from data processing you've done before?")

In [11]:
#@title 📂 Upload or Use Sample Data
import pandas as pd, io
from IPython.display import display
try:
    from google.colab import files
    USING_COLAB = True
except Exception:
    USING_COLAB = False
def load_user_csv():
    if USING_COLAB:
        up = files.upload();
        if not up: raise RuntimeError('No file selected')
        fname = next(iter(up.keys()))
        return pd.read_csv(io.BytesIO(up[fname]))
    return pd.read_csv('yelp_reviews.csv')
try:
    df = load_user_csv()
except Exception:
    df = pd.DataFrame([
        ('Amazing food and friendly staff!', 'positive'),
        ('Terrible service. Never coming back.', 'negative'),
        ('Great ambiance, okay prices.', 'positive'),
        ('Food was cold and bland.', 'negative'),
        ('Loved the desserts!', 'positive'),
        ('Waited an hour for a table.', 'negative'),
    ], columns=['text','sentiment'])
display(df.head()); print('Rows:', len(df))

Saving simple_yelp_reviews.csv to simple_yelp_reviews.csv


,text,sentiment
0,what the heck. I did smog other car from Jiffy...,negative
1,I always in the Kalbi Plate with Kim Chi. Chic...,positive
2,The chicken shwarma is awesome! The fries are ...,positive
3,Came in on a thursday lunch. the place was fil...,positive
4,I liked it..The patio was very pleasant and th...,positive


Rows: 10000


## 2) Clean and normalize your Yelp dataset
---
Raw data is messy. Before we train, we’ll make sure the CSV is consistent and ready.

We expect a CSV with **two columns**:
- **text** — the review text
- **sentiment** — the target label (e.g., `positive` / `negative`, or variants like `pos/neg`, `1/0`, `true/false`)

Our cleaning goals:
- Normalize column names to lowercase
- Trim whitespace
- Accept common label variants and map them to **{positive, negative}**
- Drop empty rows and duplicates
- Give a quick **class distribution** check

---

### Why this matters
- Models are picky: `" Positive "` vs `"positive"` should be treated the same.
- Inconsistent labels (e.g., `pos`, `1`, `true`) should map to the same meaning.
- Duplicates and empty text can skew training and waste compute.

---

In [ ]:
#@title Reflection: Train/Test Split
reflection_box("What do you think this code is doing? Why do you think we split the data into two groups?")

In [ ]:
#@title Reflection: Stratification
reflection_box("What do you think 'stratify=y' means? Why might it be important?")

In [ ]:
#@title Reflection: Why Split Data?
reflection_box("Why do you think we don't use the same data for both training and testing?")

In [ ]:
#@title 🧽 Clean dataset
import pandas as pd
valid = {'positive','negative','pos','neg','1','0','true','false'}
df = df.rename(columns={c:c.lower() for c in df.columns})
assert {'text','sentiment'}.issubset(df.columns), 'CSV must have columns: text,sentiment'
df['text'] = df['text'].astype(str).str.strip()
df['sentiment'] = df['sentiment'].astype(str).str.strip().str.lower()
df = df[df['text']!='']
df = df[df['sentiment'].isin(valid)]
map_label = {'positive':'positive','pos':'positive','1':'positive','true':'positive',
             'negative':'negative','neg':'negative','0':'negative','false':'negative'}
df['sentiment'] = df['sentiment'].map(map_label)
df = df.drop_duplicates(subset=['text','sentiment']).reset_index(drop=True)
print(df['sentiment'].value_counts())

## 3) Split train/test
---
Before we can teach the computer, we need to divide our data into two groups:
- Training data: what the computer will "study"

- Testing data: what the computer will later "quiz" itself on
---

### Why this matters:
- If we trained and tested on the same data, the model could just memorize answers.
- By splitting, we ensure that testing truly checks how well the model generalizes.

---

To do this, we need to use a helpful function from **scikit-learn** (a Python ML library).

Your job is to help us:


*   **Find out what that function is**
*   **write the import statement and the function call**

---

### Hints
- The function you need lives in `sklearn.model_selection`.  
- Its name starts with **train** and ends with **split**.  
- `test_size=0.2` means 20% of the data will be set aside for testing.  
- `random_state=42` ensures the same split every time (for reproducibility).  
- `stratify=y` makes sure both sets keep the same balance of positive/negative examples.

---


In [ ]:
#@title Train/test split Fill in the blank
import numpy as np
from sklearn.model_selection import train_test_split
X = np.asarray(df['text']); y = np.asarray(df['sentiment'])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    random_state=42, stratify=y)
print(len(X_train), len(X_test))

In [ ]:
#@title Reflection: Understanding TF-IDF
reflection_box("What do you think TF-IDF is doing? What do you think 'Term Frequency' and 'Inverse Document Frequency' might mean?")

In [ ]:
#@title Reflection: Converting Text to Numbers
reflection_box("What do you think happens when we convert text to numbers? Why do you think this is necessary?")

In [ ]:
#@title Reflection: Text to Numbers Process
reflection_box("How is converting text to numbers similar to or different from other data transformations you've seen?")

In [ ]:
#@title ✅ Train/test split — Solution
import numpy as np
# This import brings in the function that automatically splits our data
from sklearn.model_selection import train_test_split

# Convert columns to NumPy arrays (a common ML format)
X = np.asarray(df['text'])        # Features → the actual review text
y = np.asarray(df['sentiment'])   # Labels → positive or negative sentiment

# Split the dataset:
# - 80% goes into X_train/y_train for learning
# - 20% goes into X_test/y_test for evaluation
# - random_state=42 ensures consistency every time you run this cell
# - stratify=y keeps the same positive/negative ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Print out the sizes to confirm the split worked
print("Training samples:", len(X_train), " | Testing samples:", len(X_test))

In [ ]:
#@title Reflection: Model Training
reflection_box("What do you think happens when we call .fit()? What do you think the model is learning?")

In [ ]:
#@title Reflection: What is a Classifier?
reflection_box("What do you think a 'classifier' does? What do you think LinearSVC might stand for?")

In [ ]:
#@title Reflection: Training vs Explicit Rules
reflection_box("How is training a model different from writing explicit rules (like if-else statements) to classify text?")

## 4) Turn text into numbers (TF-IDF)
---
Computers can’t understand words directly.  
So we use **TF-IDF (Term Frequency–Inverse Document Frequency)** to turn words into numbers that capture how important each word is across all reviews.  

Example:  
- "Amazing food and friendly staff" → [0.8, 0.0, 0.4, ...]  
- "Terrible food and rude service" → [0.0, 0.9, 0.3, ...]  

TF-IDF gives higher weight to words that are common in one review but rare overall.

---

### Step 1: Choose your n-grams  
We can look at:
- **Single words (unigrams)** like “great”
- **Pairs of words (bigrams)** like “not great”

That’s what `ngram_range=(1, 2)` means — it includes both unigrams and bigrams.

---

### Step 2: Build the converter  
We’ll use a TF-IDF vectorizer to:
1. **fit + transform** the training text (`X_train`) → learn the vocabulary and convert to numbers  
2. **transform** the test text (`X_test`) → use the same vocabulary to convert  

This ensures the model only learns from training data but can evaluate new data.

---

### What you need to do  
1. Import the TF-IDF vectorizer from the correct module.  
   (Hint: it lives in `sklearn.feature_extraction.text`.)  
2. Create a vectorizer that uses **unigrams + bigrams** and ignores super-rare words (`min_df=2`).  
3. Fit it on `X_train` and transform both `X_train` and `X_test`.  
4. Print out the resulting shapes — number of samples and features.  

---

In [ ]:
#@title Reflection: Understanding Evaluation Metrics
reflection_box("What do you think these metrics (accuracy, precision, recall) are measuring?")

### 💡 Hint: What to Import

To convert text into numbers using TF-IDF, you need to import a class from scikit-learn.

**What you need:**
- **Module location:** `sklearn.feature_extraction.text`
- **Class name:** Starts with `Tfidf` and ends with `Vectorizer`
- **Full name:** `TfidfVectorizer`

This class will convert your text reviews into numerical feature vectors that machine learning models can understand.

---

In [ ]:
#@title Reflection: Why Multiple Metrics?
reflection_box("Why might accuracy alone not be enough to understand how well a model works?")

In [ ]:
#@title 📚 Import Example
# Here's how to import TfidfVectorizer:
from sklearn.feature_extraction.text import TfidfVectorizer

# Now you can use TfidfVectorizer in the next cell!
print("✅ Import successful! You can use TfidfVectorizer now.")

In [ ]:
#@title Reflection: Making Predictions
reflection_box("What do you think happens when you type a new review and click 'Predict'? How do you think the model uses what it learned?")

In [ ]:
#@title Fill in the imports you discovered

''' from [library_path] import [ClassName] '''

# Text → numbers (TF-IDF)
# from _________________________________ import ____________________
from sklearn.feature_extraction.text import TfidfVectorizer

# Create a TF-IDF converter (unigrams + bigrams)
# Note: min_df=1 works better with small datasets; use min_df=2 for larger datasets
# vectorizer = ____________________(ngram_range=(1, 2), min_df=1, stop_words='english')
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1, stop_words='english')

# Fit on training data, then transform both train and test sets
# X_train_tfidf = vectorizer.______________(X_train)
X_train_tfidf = vectorizer.fit_transform(X_train)
# X_test_tfidf = vectorizer.______________(X_test)
X_test_tfidf = vectorizer.transform(X_test)

# Print matrix sizes to check
print("Train shape:", X_train_tfidf.shape, "Test shape:", X_test_tfidf.shape)


In [ ]:
#@title Reflection: Using Trained Models
reflection_box("How is using a trained model to make predictions different from writing explicit rules to classify text?")

In [ ]:
#@title ✅ TF-IDF vectorization — Solution
# Purpose of this import:
# TfidfVectorizer converts raw text into numeric feature vectors using TF-IDF.
from sklearn.feature_extraction.text import TfidfVectorizer

# Unigrams + bigrams; ignore terms that occur in fewer than 2 documents; remove English stop words
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1, stop_words='english')

# Fit the vectorizer on training data, then transform train and test
X_train_tfidf = vectorizer.fit_transform(X_train)   # learn vocab + weights, then transform
X_test_tfidf = vectorizer.transform(X_test)        # transform using the same learned vocab

print("Train shape:", X_train_tfidf.shape, " Test shape:", X_test_tfidf.shape)


## 5) Train models

In [ ]:
#@title 🤖 Fit classifiers
from sklearn.svm import LinearSVC
clf = LinearSVC()
clf.fit(X_train_tfidf, y_train)
print('Trained: LinearSVC')

## 6) Evaluate your model(s)
---
Training is only half the story — now we check how well the model learned.

We’ll compute:
- **Accuracy** — % of reviews predicted correctly.
- **Precision/Recall/F1** — quality of predictions for each class (positive/negative).
- **Confusion Matrix** — a 2×2 table that compares *true* vs *predicted* labels.

---

### Why this matters
- A high accuracy alone can be misleading if the dataset is imbalanced.
- Precision/Recall/F1 help you understand *how* the model is making mistakes.
- The confusion matrix shows **where** those mistakes happen (e.g., positive → negative).

---

In [ ]:
#@title 📈 Evaluation
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt, numpy as np, itertools
import seaborn as sns
def plot_cm(y_true, y_pred, title='Confusion Matrix'):
    cm = confusion_matrix(y_true, y_pred, labels=['negative','positive'])
    classes = ['negative','positive']
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=["neg", "pos"], yticklabels=["neg", "pos"])
    plt.title(title)
    plt.xlabel('Predicted'); plt.ylabel('True')
    plt.tight_layout(); plt.show()
y_pred = clf.predict(X_test_tfidf)
acc = accuracy_score(y_test, y_pred)
print("Accuracy:", acc)
print(classification_report(y_test, y_pred, target_names=["negative", "positive"]))
plot_cm(y_test, y_pred, 'LinearSVC Confusion Matrix')

## 7) Try it

In [ ]:
#@title ✍️ Predict your own review
import ipywidgets as widgets
from IPython.display import display
box = widgets.Textarea(placeholder='Type a short Yelp-style review...')
btn = widgets.Button(description='Predict')
out = widgets.Output()
def go(_):
    Xq = vectorizer.transform([box.value])
    prediction = clf.predict(Xq)[0]
    with out:
        out.clear_output(); print('LinearSVC prediction:', prediction)
btn.on_click(go); display(box, btn, out)

## 8) Save best model

In [ ]:
#@title 💾 Save model
import joblib
joblib.dump({'model': clf, 'vectorizer': vectorizer}, 'yelp_sentiment.joblib')
print('Saved as yelp_sentiment.joblib')